# 01. PT2E Export Quantization Workflow: Conv-BN Folding, Q/DQ Lowering, and Hardware Backend Targeting

- **Category:** 14_quantisation / 02_pt2e_torchao_and_bitpacking
- **Lineage & Citations:** Jacob et al. (CVPR 2018, *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*); PyTorch Core & Edge Team (2024, *PyTorch 2 Export Quantization PT2E Architecture & XNNPACKQuantizer Specification*).
- **Purpose:** Formulate the PyTorch 2 Export Quantization (PT2E) workflow, algebraic Conv2d-BatchNorm2d and Linear-BatchNorm1d operator folding, simulated Quantize-Dequantize (Q/DQ) graph lowering, and hardware-aligned operator fusion.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. Calibration Observers](../01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.ipynb) $\to$ `🔗 Downstream Syntheses:` [02. TorchAO Modern API and Affine Quantized Tensor](02_torchao_modern_api_and_affine_quantized_tensor.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT quantize unfused `Conv2d + BatchNorm2d` (BatchNorm's $1/\sigma_{\text{BN}}$ multiplier amplifies INT8 quantization noise by $>10\times$ unless folded into Conv weights *before* quantization!); do NOT export models in `train()` mode for PT2E.

## Mathematical Foundations & Conv-BatchNorm Algebraic Fusion

1. **Unfolded Sequential Composition**:
   $$y_{\text{conv}} = x * W_{\text{conv}} + b_{\text{conv}}, \quad y_{\text{bn}} = \frac{y_{\text{conv}} - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$
2. **Algebraic Fusion into Single Equivalent Operator**:
   $$s_c = \frac{\gamma_c}{\sqrt{\sigma_c^2 + \epsilon}}, \quad W_{\text{folded}}[c] = W_{\text{conv}}[c] \cdot s_c, \quad b_{\text{folded}}[c] = (b_{\text{conv}}[c] - \mu_c) \cdot s_c + \beta_c$$
3. **Quantize-Dequantize (Q/DQ) Graph Lowering**:
   $$Q(x; s) = \operatorname{clamp}\left(\left\lfloor \frac{x}{s} \right\rceil, -128, 127\right) \cdot s, \quad \hat{y} = Q(x; s_x) * Q(W_{\text{folded}}; s_w) + b_{\text{folded}}$$
4. **Unfused Quantization Noise Explosion**:
   $$\text{Error}_{\text{unfused}} = \Delta W \cdot s_c \gg \Delta W_{\text{folded}} \implies \text{SNR degrades by } 20 \log_{10}(s_c) \text{ dB}.

In [ ]:
import sys, time, math
from pathlib import Path
import torch
import torch.nn.functional as F
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/14_quantisation/02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def conv2d_bn_fold(w_conv, b_conv, bn_mean, bn_var, bn_gamma, bn_beta, eps=1e-5):
    scale = bn_gamma / torch.sqrt(bn_var + eps)
    w_folded = w_conv * scale.view(-1, 1, 1, 1)
    b_folded = (b_conv - bn_mean) * scale + bn_beta
    return w_folded, b_folded

def linear_bn_fold(w_linear, b_linear, bn_mean, bn_var, bn_gamma, bn_beta, eps=1e-5):
    scale = bn_gamma / torch.sqrt(bn_var + eps)
    w_folded = w_linear * scale.unsqueeze(0)
    b_folded = (b_linear - bn_mean) * scale + bn_beta
    return w_folded, b_folded

def quantize_symmetric_int8(x, scale, qmin=-128.0, qmax=127.0):
    if scale == 0.0: return torch.zeros_like(x)
    return torch.clamp(torch.round(x / scale), qmin, qmax) * scale

def pt2e_qdq_conv2d(x, w_folded, b_folded, s_x, s_w):
    x_q = quantize_symmetric_int8(x, s_x)
    w_q = quantize_symmetric_int8(w_folded, s_w)
    return F.conv2d(x_q, w_q, b_folded)

def pt2e_qdq_linear(x, w_folded, b_folded, s_x, s_w):
    x_q = quantize_symmetric_int8(x, s_x)
    w_q = quantize_symmetric_int8(w_folded, s_w)
    return x_q @ w_q + b_folded

def unfused_conv_bn_quantize(x, w_conv, b_conv, bn_mean, bn_var, bn_gamma, bn_beta, s_w, eps=1e-5):
    w_q = quantize_symmetric_int8(w_conv, s_w)
    conv_out = F.conv2d(x, w_q, b_conv)
    scale = bn_gamma / torch.sqrt(bn_var + eps)
    return (conv_out - bn_mean.view(1, -1, 1, 1)) * scale.view(1, -1, 1, 1) + bn_beta.view(1, -1, 1, 1)

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. Conv-BN Folding Parity
wf, bf = conv2d_bn_fold(
    weights["w_conv"], weights["b_conv"], weights["bn_mean"], weights["bn_var"], weights["bn_gamma"], weights["bn_beta"]
)
assert torch.max(torch.abs(wf - weights["w_folded"])) < 1e-5
assert torch.max(torch.abs(bf - weights["b_folded"])) < 1e-5

# 3. Linear-BN Folding Parity
wlf, blf = linear_bn_fold(
    weights["w_lin"], weights["b_lin"], weights["lin_mean"], weights["lin_var"], weights["lin_gamma"], weights["lin_beta"]
)
assert torch.max(torch.abs(wlf - weights["w_lin_folded"])) < 1e-5
assert torch.max(torch.abs(blf - weights["b_lin_folded"])) < 1e-5

# 4. QDQ Conv2d Parity
s_x_c = float(torch.max(torch.abs(weights["x_conv"]))) / 127.0
s_w_c = float(torch.max(torch.abs(weights["w_folded"]))) / 127.0
out_qdq_c = pt2e_qdq_conv2d(weights["x_conv"], weights["w_folded"], weights["b_folded"], s_x_c, s_w_c)
assert torch.max(torch.abs(out_qdq_c - weights["out_qdq_conv"])) < 1e-5

# 5. QDQ Linear Parity
s_x_l = float(torch.max(torch.abs(weights["x_lin"]))) / 127.0
s_w_l = float(torch.max(torch.abs(weights["w_lin_folded"]))) / 127.0
out_qdq_l = pt2e_qdq_linear(weights["x_lin"], weights["w_lin_folded"], weights["b_lin_folded"], s_x_l, s_w_l)
assert torch.max(torch.abs(out_qdq_l - weights["out_qdq_lin"])) < 1e-5

# 6. Unfused Conv-BN Parity
out_unf = unfused_conv_bn_quantize(
    weights["x_conv"], weights["w_conv"], weights["b_conv"], weights["bn_mean"], weights["bn_var"], weights["bn_gamma"], weights["bn_beta"], s_w_c
)
assert torch.max(torch.abs(out_unf - weights["out_unfused"])) < 1e-5
print("All PT2E Conv-BN and Linear-BN folding parity assertions verified!")

In [ ]:
# Microbenchmark: Unfolded Conv2d + BatchNorm2d (2 passes) vs Folded Conv2d (1 pass)
x_bench = weights["x_conv"].repeat(10, 1, 1, 1)
t0 = time.perf_counter()
for _ in range(100):
    c_out = F.conv2d(x_bench, weights["w_conv"], weights["b_conv"])
    scale = weights["bn_gamma"] / torch.sqrt(weights["bn_var"] + 1e-5)
    _ = (c_out - weights["bn_mean"].view(1, -1, 1, 1)) * scale.view(1, -1, 1, 1) + weights["bn_beta"].view(1, -1, 1, 1)
t_unfolded = (time.perf_counter() - t0) / 100 * 1e3

t0 = time.perf_counter()
for _ in range(100):
    _ = F.conv2d(x_bench, weights["w_folded"], weights["b_folded"])
t_folded = (time.perf_counter() - t0) / 100 * 1e3

print(f"Unfolded Conv+BN latency: {t_unfolded:.2f} ms (2 kernel dispatches + DRAM roundtrip)")
print(f"Fused Folded Conv2d:      {t_folded:.2f} ms ({t_unfolded / t_folded:.1f}x speedup!)")

In [ ]:
# Failure Mode: Unfused Quantization Noise Explosion vs Pre-Folded Quantization
# Ideal FP32 baseline (ground truth)
fp32_baseline = F.conv2d(weights["x_conv"], weights["w_folded"], weights["b_folded"])

# 1. Pre-Folded PT2E Quantized Conv2d
out_folded_q = pt2e_qdq_conv2d(weights["x_conv"], weights["w_folded"], weights["b_folded"], s_x_c, s_w_c)
mse_folded = float(torch.mean((fp32_baseline - out_folded_q) ** 2))
sqnr_folded = 10.0 * math.log10(float(torch.mean(fp32_baseline ** 2)) / mse_folded)

# 2. Unfused Quantization (Conv quantized before BN, amplifying noise by 1/sigma)
mse_unfused = float(torch.mean((fp32_baseline - out_unf) ** 2))
sqnr_unfused = 10.0 * math.log10(float(torch.mean(fp32_baseline ** 2)) / mse_unfused)

print(f"Pre-Folded PT2E SQNR:  {sqnr_folded:.2f} dB (Quantization bounds true output error)")
print(f"Unfused Naive SQNR:    {sqnr_unfused:.2f} dB (BatchNorm multiplier amplifies error!)")
sqnr_advantage = sqnr_folded - sqnr_unfused
print(f"Folding SQNR Advantage: +{sqnr_advantage:.2f} dB over naive unfused quantization!")
assert sqnr_advantage > 1.0, "Pre-folding must outperform unfused quantization in SQNR!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Graph-Level Algebraic Fusion**: Pre-folding normalization into convolution and linear weights eliminates runtime overhead and prevents $1/\sigma_{\text{BN}}$ quantization noise amplification.
2. **Functional PT2E Lowering**: `prepare_pt2e` inserts explicit Q/DQ nodes into the ATen FX graph, enabling backend quantizers (`XNNPACKQuantizer`) to annotate hardware-aligned scale invariants.
3. **Zero-Overhead Edge Deployment**: Lowering to fused quantized kernels (`convert_pt2e`) removes floating-point conversion roundtrips, maximizing integer SIMD/TensorCore execution throughput.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_pt2e_export_quantization_workflow.md](../../../markdown/14_quantisation/02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.md)
- **PyTorch 2.x Module:** [01_pt2e_export_quantization_workflow.py](../../../pytorch2/14_quantisation/02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.py)
- **Functional JAX Module:** [01_pt2e_export_quantization_workflow.py](../../../jax/14_quantisation/02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.py)
- **Native Rust Candle Track:** [01_pt2e_export_quantization_workflow.rs](../../../candle/14_quantisation/02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.rs)
- **Hardware Kernel (CPU / WGSL):** [01_pt2e_export_quantization_workflow.rs](../../../kernels/14_quantisation/02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.rs)
- **Golden SafeTensors:** [01_pt2e_export_quantization_workflow.safetensors](../../../weights/14_quantisation/02_pt2e_torchao_and_bitpacking/01_pt2e_export_quantization_workflow.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Calibration Observers](../01_foundations_formats_and_observers/04_calibration_observers_minmax_percentile_mse_kl_and_aciq.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 02. TorchAO Modern API and Affine Quantized Tensor](02_torchao_modern_api_and_affine_quantized_tensor.ipynb) &rarr;